# Flat Manifold Vector Fields

Playground notebook for pure data generation. The reusable generator lives in `simulation/flat_manifold_vector_fields.py`, and names follow `flat_manifold__<field>_vector_field`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
while not (ROOT / "simulation").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from simulation.flat_manifold_vector_fields import (
    FIELD_NAMES,
    FlatVectorFieldConfig,
    make_all_flat_manifold_vector_fields,
    make_flat_manifold_vector_field,
    save_npz,
)

FIELD_NAMES

In [ ]:
def plot_vector_field(simulation, ax=None, max_arrows=400, seed=42, title_suffix=""):
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))

    rng = np.random.default_rng(seed)
    x = simulation["X"][:, :2]
    v = simulation["V"][:, :2]
    time = simulation["true_time"]
    time = (time - time.min()) / (time.max() - time.min() + 1e-12)

    n_arrows = min(max_arrows, len(x))
    idx = rng.choice(len(x), size=n_arrows, replace=False)

    ax.scatter(x[:, 0], x[:, 1], c=time, cmap="viridis", s=10, alpha=0.25)
    ax.quiver(
        x[idx, 0], x[idx, 1],
        v[idx, 0], v[idx, 1],
        color=plt.cm.viridis(time[idx]),
        angles="xy",
        scale_units="xy",
        scale=5,
        alpha=0.75,
        width=0.003,
    )
    ax.set_title(f"{simulation['config']['simulation_name']}{title_suffix}")
    ax.set_aspect("equal")
    ax.axis("off")
    return ax

## One Simulation

Change `field_name`, `position_noise`, `velocity_noise`, `extra_dims`, `n_samples`, and `seed` while iterating.

In [ ]:
config = FlatVectorFieldConfig(
    field_name="rotation",
    n_samples=1000,
    position_noise=0.3,
    velocity_noise=0.3,
    extra_dims=5,
    seed=42,
)

simulation = make_flat_manifold_vector_field(config)
simulation["config"]

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
plot_vector_field(simulation, ax=ax)
plt.show()

## Field Grid

This is the old flat-field view without fitting: one panel per vector field.

In [ ]:
simulations = make_all_flat_manifold_vector_fields(
    n_samples=1000,
    position_noise=0.3,
    velocity_noise=0.3,
    extra_dims=5,
    seed=42,
)

fig, axes = plt.subplots(1, len(simulations), figsize=(5 * len(simulations), 5))
for ax, simulation in zip(axes, simulations.values()):
    plot_vector_field(simulation, ax=ax)
plt.tight_layout()
plt.show()

## Save Data

Uncomment when a generated dataset is worth keeping.

In [ ]:
# path = save_npz(simulation)
# path